# AP02 - Advanced Indexing Strategies

### Master different index types and optimization techniques

**Duration:** 2.5 hours

**What you'll learn:**
- Different index types (Vector, List, Tree, Keyword, Knowledge Graph)
- When to use each index type
- Composable indexes for complex scenarios
- Performance optimization strategies
- Hybrid search combining multiple approaches
- Index persistence and caching

**Prerequisites:**
- Completed AP01 - Introduction to LlamaIndex
- Understanding of vector similarity search

---

## 🔧 Setup

In [ ]:
# Install dependencies
# !pip install -qU llama-index llama-index-llms-openai llama-index-embeddings-openai \
#     llama-index-graph-stores-networkx python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# For Google Colab
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY"
print("✅ Setup complete")

---

## 1. Index Types Overview

### LlamaIndex Index Types

| Index Type | Best For | Speed | Accuracy | Use Case |
|------------|----------|-------|----------|----------|
| **VectorStoreIndex** | Semantic search | Fast | High | General Q&A, RAG |
| **ListIndex** | Sequential scan | Slow | High | Small datasets, completeness |
| **TreeIndex** | Hierarchical data | Medium | Medium | Summarization, hierarchies |
| **KeywordTableIndex** | Exact matching | Very Fast | Medium | Keyword search |
| **KnowledgeGraphIndex** | Relationships | Medium | High | Connected data, reasoning |

---

## 2. VectorStoreIndex Deep Dive

### Default and Most Common

In [ ]:
from llama_index.core import VectorStoreIndex, Document

# Create documents about programming languages
docs = [
    Document(text="Python is a high-level, interpreted language known for readability."),
    Document(text="JavaScript is essential for web development and runs in browsers."),
    Document(text="Rust provides memory safety without garbage collection."),
    Document(text="Go is designed for concurrent programming and scalability."),
]

# Build vector index (uses embeddings)
vector_index = VectorStoreIndex.from_documents(docs)

# Query
query_engine = vector_index.as_query_engine(similarity_top_k=2)
response = query_engine.query("Which language is good for web development?")

print("Response:", response)
print("\nSources:")
for i, node in enumerate(response.source_nodes):
    print(f"{i+1}. {node.text[:60]}... (score: {node.score:.3f})")

### Customizing VectorStoreIndex

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding

# Use different embedding model
Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-large",  # More powerful but slower
    dimensions=1024  # Can reduce dimensions for speed
)

# Customize chunk size
Settings.chunk_size = 256
Settings.chunk_overlap = 25

# Build with custom settings
custom_index = VectorStoreIndex.from_documents(docs)

print(f"✅ Custom index created")
print(f"   Embed model: {Settings.embed_model.model_name}")
print(f"   Chunk size: {Settings.chunk_size}")

---

## 3. ListIndex - Sequential Processing

In [ ]:
from llama_index.core import ListIndex

# Create ListIndex - processes ALL documents sequentially
list_index = ListIndex.from_documents(docs)

# Query with ListIndex
list_query_engine = list_index.as_query_engine(
    response_mode="tree_summarize"  # Good for comprehensive answers
)

response = list_query_engine.query("Compare all programming languages mentioned")
print("ListIndex Response:")
print(response)

### When to Use ListIndex

✅ **Use ListIndex when:**
- Small number of documents (<50)
- Need comprehensive answers using ALL documents
- Quality over speed
- Want to avoid missing relevant info

❌ **Avoid ListIndex when:**
- Large datasets (>100 docs)
- Need fast responses
- Only need top-k relevant results

---

## 4. TreeIndex - Hierarchical Summarization

In [ ]:
from llama_index.core import TreeIndex

# Create longer documents for better demonstration
long_docs = [
    Document(text="""Python Overview: Python is a versatile, high-level programming language.
    It features dynamic typing, automatic memory management, and extensive standard library.
    Python excels in data science, machine learning, web development, and automation.
    Its syntax emphasizes readability with significant whitespace."""),
    
    Document(text="""JavaScript Ecosystem: JavaScript powers modern web applications.
    With Node.js, it runs on servers. React, Vue, and Angular dominate frontend.
    NPM provides the largest package ecosystem. TypeScript adds static typing.
    JavaScript is essential for full-stack development."""),
    
    Document(text="""Rust for Systems Programming: Rust guarantees memory safety without garbage collection.
    Its ownership system prevents data races at compile time.
    Rust is ideal for systems programming, embedded systems, and performance-critical applications.
    Growing adoption in web assembly and blockchain development.""")
]

# Build TreeIndex
tree_index = TreeIndex.from_documents(long_docs)

# Query for summary
tree_query_engine = tree_index.as_query_engine()
response = tree_query_engine.query("Summarize the key strengths of each language")

print("TreeIndex Summary:")
print(response)

### When to Use TreeIndex

✅ **Use TreeIndex when:**
- Need hierarchical summarization
- Documents have natural hierarchy
- Building summaries at different levels
- Synthesizing information from many sources

---

## 5. KeywordTableIndex - Exact Matching

In [ ]:
from llama_index.core import KeywordTableIndex

# Create documents with distinct keywords
keyword_docs = [
    Document(
        text="Django is a high-level Python web framework.",
        metadata={"keywords": ["django", "python", "web", "framework"]}
    ),
    Document(
        text="React is a JavaScript library for building user interfaces.",
        metadata={"keywords": ["react", "javascript", "ui", "library"]}
    ),
    Document(
        text="TensorFlow is a machine learning framework.",
        metadata={"keywords": ["tensorflow", "ml", "framework", "google"]}
    )
]

# Build KeywordTableIndex
keyword_index = KeywordTableIndex.from_documents(keyword_docs)

# Query with exact keywords
keyword_query_engine = keyword_index.as_query_engine()
response = keyword_query_engine.query("framework")

print("Keyword Search Results:")
print(response)
print("\nMatched on keyword: 'framework'")

### When to Use KeywordTableIndex

✅ **Use KeywordTableIndex when:**
- Need exact keyword matching
- Have well-defined keywords
- Speed is critical
- Combining with vector search (hybrid)

---

## 6. KnowledgeGraphIndex - Relationship-Based

In [ ]:
from llama_index.core import KnowledgeGraphIndex
from llama_index.core.graph_stores import SimpleGraphStore

# Create documents with relationships
kg_docs = [
    Document(text="Guido van Rossum created Python in 1991."),
    Document(text="Python is used by Google, Netflix, and NASA."),
    Document(text="Brendan Eich created JavaScript in 1995."),
    Document(text="JavaScript runs in all modern web browsers."),
]

# Build Knowledge Graph
graph_store = SimpleGraphStore()
kg_index = KnowledgeGraphIndex.from_documents(
    kg_docs,
    graph_store=graph_store,
    max_triplets_per_chunk=3
)

# Query for relationships
kg_query_engine = kg_index.as_query_engine(
    include_text=True,
    response_mode="tree_summarize"
)

response = kg_query_engine.query("Who created Python and where is it used?")
print("Knowledge Graph Response:")
print(response)

### When to Use KnowledgeGraphIndex

✅ **Use KnowledgeGraphIndex when:**
- Data has explicit relationships
- Need to answer "who, what, when, where" questions
- Building knowledge bases
- Reasoning over connections

---

## 7. Composable Indexes - Best of Multiple Worlds

In [ ]:
from llama_index.core import ComposableGraph

# Create different indexes for different data
tech_docs = [
    Document(text="AI and machine learning are transforming industries."),
    Document(text="Cloud computing provides scalable infrastructure.")
]

people_docs = [
    Document(text="Andrew Ng is a pioneer in machine learning education."),
    Document(text="Yann LeCun invented convolutional neural networks.")
]

# Build separate indexes
tech_index = VectorStoreIndex.from_documents(tech_docs)
people_index = VectorStoreIndex.from_documents(people_docs)

# Create summaries for routing
tech_summary = "Information about technology trends and concepts"
people_summary = "Information about people and their contributions"

# Compose them
graph = ComposableGraph.from_indices(
    ListIndex,
    [tech_index, people_index],
    index_summaries=[tech_summary, people_summary]
)

# Query - automatically routes to right index
query_engine = graph.as_query_engine()

# Test different queries
print("Query 1: Technology")
response1 = query_engine.query("What is transforming industries?")
print(response1)

print("\n" + "="*70)
print("Query 2: People")
response2 = query_engine.query("Who invented CNNs?")
print(response2)

### Composable Index Use Cases

1. **Multi-domain Knowledge Base**: Different indexes for different topics
2. **Multi-format Data**: PDFs in one index, code in another
3. **Performance Optimization**: Fast keyword index + accurate vector index
4. **Hybrid Search**: Combine semantic + keyword + graph

---

## 8. Hybrid Search - Combining Approaches

In [ ]:
from llama_index.core import VectorStoreIndex, KeywordTableIndex
from llama_index.core.indices.query.query_transform import HyPothesisQueryTransform
from llama_index.core.query_engine import RetrieverQueryEngine

# Sample documents
hybrid_docs = [
    Document(
        text="Python is great for data science and machine learning projects.",
        metadata={"keywords": ["python", "data science", "ml"]}
    ),
    Document(
        text="JavaScript is essential for modern web development.",
        metadata={"keywords": ["javascript", "web", "frontend"]}
    ),
    Document(
        text="Rust provides memory safety and high performance.",
        metadata={"keywords": ["rust", "systems", "performance"]}
    )
]

# Build both indexes
vector_idx = VectorStoreIndex.from_documents(hybrid_docs)
keyword_idx = KeywordTableIndex.from_documents(hybrid_docs)

# Create hybrid retriever (simplified approach)
vector_retriever = vector_idx.as_retriever(similarity_top_k=2)
keyword_retriever = keyword_idx.as_retriever()

# Query both
query = "Which language for machine learning?"

print("Vector Search Results:")
vector_nodes = vector_retriever.retrieve(query)
for node in vector_nodes:
    print(f"- {node.text} (score: {node.score:.3f})")

print("\nKeyword Search Results:")
keyword_nodes = keyword_retriever.retrieve("ml machine learning")
for node in keyword_nodes:
    print(f"- {node.text}")

print("\n💡 Hybrid search combines both for better results!")

---

## 9. Performance Optimization

### Strategy 1: Chunk Size Optimization

In [ ]:
from llama_index.core import Settings
import time

# Create test document
test_docs = [
    Document(text=" ".join(["This is test sentence number " + str(i) + "." for i in range(100)]))
]

# Test different chunk sizes
chunk_sizes = [128, 256, 512, 1024]

for chunk_size in chunk_sizes:
    Settings.chunk_size = chunk_size
    Settings.chunk_overlap = chunk_size // 10
    
    start = time.time()
    index = VectorStoreIndex.from_documents(test_docs)
    elapsed = time.time() - start
    
    print(f"Chunk size {chunk_size}: {elapsed:.3f}s")

print("\n💡 Smaller chunks = faster indexing, more chunks")
print("💡 Larger chunks = slower indexing, better context")

### Strategy 2: Caching

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage
import os

persist_dir = "./storage/cache_example"

# First time: Build and save
if not os.path.exists(persist_dir):
    print("Building index (first time)...")
    start = time.time()
    index = VectorStoreIndex.from_documents(docs)
    index.storage_context.persist(persist_dir=persist_dir)
    print(f"Built and saved in {time.time() - start:.3f}s")
else:
    print("Loading cached index...")
    start = time.time()
    storage_context = StorageContext.from_defaults(persist_dir=persist_dir)
    index = load_index_from_storage(storage_context)
    print(f"Loaded in {time.time() - start:.3f}s")

print("\n💡 Caching can speed up loading by 10-100x!")

### Strategy 3: Batch Processing

In [ ]:
# Instead of adding documents one by one
# BAD:
# for doc in documents:
#     index.insert(doc)  # Many API calls

# GOOD:
index = VectorStoreIndex.from_documents(docs)  # Batch processing

# Or for updates:
index.refresh(docs)  # Incremental update

print("✅ Batch processing is faster and cheaper!")

---

## 10. Real-World Example: Multi-Index Enterprise Search

In [ ]:
# Simulate enterprise data
product_docs = [
    Document(text="Product X1000: High-performance laptop with 32GB RAM", 
             metadata={"category": "hardware", "price": 1500}),
    Document(text="Product S500: Ergonomic office chair with lumbar support",
             metadata={"category": "furniture", "price": 400})
]

policy_docs = [
    Document(text="Return policy: 30-day money-back guarantee on all products",
             metadata={"type": "policy"}),
    Document(text="Shipping: Free shipping on orders over $50",
             metadata={"type": "policy"})
]

faq_docs = [
    Document(text="Q: How do I track my order? A: Use the tracking link in your email",
             metadata={"type": "faq"}),
    Document(text="Q: What payment methods do you accept? A: Credit cards, PayPal, and bank transfer",
             metadata={"type": "faq"})
]

# Build specialized indexes
product_index = VectorStoreIndex.from_documents(product_docs)
policy_index = VectorStoreIndex.from_documents(policy_docs)
faq_index = VectorStoreIndex.from_documents(faq_docs)

# Create composable graph
enterprise_graph = ComposableGraph.from_indices(
    ListIndex,
    [product_index, policy_index, faq_index],
    index_summaries=[
        "Product catalog with specifications and pricing",
        "Company policies including returns and shipping",
        "Frequently asked questions and answers"
    ]
)

# Create query engine
enterprise_engine = enterprise_graph.as_query_engine()

# Test different query types
queries = [
    "What laptops do you have?",
    "What's your return policy?",
    "How can I track my order?"
]

for query in queries:
    print(f"\n{'='*70}")
    print(f"Q: {query}")
    print(f"{'='*70}")
    response = enterprise_engine.query(query)
    print(f"A: {response}")

---

## 🎯 Practice Exercises

### Exercise 1: Index Comparison
Create the same dataset with all 5 index types. Compare:
- Build time
- Query speed
- Answer quality
- Memory usage

### Exercise 2: Hybrid Search System
Build a hybrid search combining:
- Vector search for semantic matching
- Keyword search for exact terms
- Merge and rank results

### Exercise 3: Multi-Domain Knowledge Base
Create composable index with:
- Technical documentation
- User guides
- FAQs
- Test routing to correct index

### Exercise 4: Performance Optimization
Optimize an index for:
- Large dataset (1000+ docs)
- Fast query time (<500ms)
- Measure and document improvements

### Challenge: Enterprise Search Engine
Build production-ready search with:
- Multiple data sources
- Caching strategy
- Monitoring
- Error handling
- API endpoint

---

## 📚 Key Takeaways

1. **VectorStoreIndex** - Default choice for most RAG applications
2. **ListIndex** - When you need comprehensive answers from all docs
3. **TreeIndex** - For hierarchical summarization
4. **KeywordTableIndex** - Fast exact matching
5. **KnowledgeGraphIndex** - When relationships matter
6. **Composable** - Combine indexes for complex scenarios
7. **Hybrid Search** - Best of semantic + keyword
8. **Optimization** - Chunk size, caching, batching matter

---

## 🚀 Next Steps

**AP03 - Query Engines and Retrievers**
- Custom query engines
- Advanced retrieval strategies
- Query transformations
- Response synthesis

---

*Created as part of the Applied AI module*  
*Last updated: January 2025*